# 08 - Model Explainability (SHAP)

## Objective
Answer "why" for attrition predictions:
- **Global:** What factors drive attrition across the company?
- **Local:** Why is THIS specific employee flagged as high risk?

SHAP values provide both levels of explanation, making predictions actionable for HR.
---

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import shap
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

MODEL_PATH = "../models"
DATA_PATH = "../data/processed"

# Load pipeline
pipeline = joblib.load(f"{MODEL_PATH}/attrition_pipeline.joblib")
model = pipeline["model"]
scaler = pipeline["scaler"]
feature_names = pipeline["feature_names"]
model_name = pipeline["model_name"]
print(f"Loaded model: {model_name}")

# Load data
df = pd.read_csv(f"{DATA_PATH}/attrition_features.csv")
y = df["Attrition"]
X = df.drop(columns=["Attrition"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Test set: {X_test.shape[0]} samples")

Loaded model: XGBoost
Test set: 294 samples


---
## 1. Compute SHAP Values
---

In [2]:
# Use TreeExplainer for tree-based models, LinearExplainer for linear
model_type = type(model).__name__
print(f"Model type: {model_type}")

if model_type in ["RandomForestClassifier", "XGBClassifier"]:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    # For binary classification, shap_values might be a list
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]  # class 1 = Attrition=Yes
    else:
        shap_vals = shap_values
else:
    # For linear models, use the scaled data
    X_test_scaled = scaler.transform(X_test)
    explainer = shap.LinearExplainer(model, scaler.transform(X_train))
    shap_vals = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_vals.shape}")

Model type: XGBClassifier
SHAP values shape: (294, 44)


---
## 2. Global Feature Importance (SHAP Summary)
---

In [3]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_vals, X_test, feature_names=feature_names, show=False, max_display=20)
plt.title("SHAP Summary — What Drives Attrition Globally")
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/shap_summary.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: models/shap_summary.png")

Saved: models/shap_summary.png


---
## 3. Feature Importance Bar Chart
---

In [4]:
# Mean absolute SHAP values
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP": np.abs(shap_vals).mean(axis=0)
}).sort_values("Mean_SHAP", ascending=False)

print("Top 15 features by mean |SHAP|:")
print(shap_importance.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
top15 = shap_importance.head(15)
ax.barh(range(len(top15)), top15["Mean_SHAP"].values[::-1], color="#3b82f6")
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15["Feature"].values[::-1])
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Top 15 Features — SHAP Importance")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/shap_importance_bar.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: models/shap_importance_bar.png")

Top 15 features by mean |SHAP|:
                         Feature  Mean_SHAP
                        OverTime   0.934061
                   MonthlyIncome   0.732854
                StockOptionLevel   0.561116
              NumCompaniesWorked   0.534490
                DistanceFromHome   0.523702
                             Age   0.509840
            YearsWithCurrManager   0.420181
BusinessTravel_Travel_Frequently   0.398478
              satisfaction_score   0.362002
        RelationshipSatisfaction   0.324190
         EnvironmentSatisfaction   0.303489
               PercentSalaryHike   0.296990
                 income_per_year   0.293801
                  JobInvolvement   0.277396
      JobRole_Research Scientist   0.265917


Saved: models/shap_importance_bar.png


---
## 4. Local Explanation — Specific Employee
---

In [5]:
# Pick the employee with the highest predicted attrition probability
if scaler:
    X_test_scaled = scaler.transform(X_test)
    probs = model.predict_proba(X_test_scaled)[:, 1]
else:
    probs = model.predict_proba(X_test)[:, 1]

high_risk_idx = np.argmax(probs)
high_risk_prob = probs[high_risk_idx]
high_risk_features = X_test.iloc[high_risk_idx]

print(f"High-risk employee (test index {high_risk_idx}):")
print(f"  Predicted attrition probability: {high_risk_prob:.1%}")
print(f"  Actual attrition: {'Yes' if y_test.iloc[high_risk_idx] == 1 else 'No'}")
print()
print("Top contributing factors:")
shap_vals_single = shap_vals[high_risk_idx]
factor_df = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_value": shap_vals_single,
    "Feature_value": high_risk_features.values
}).sort_values("SHAP_value", ascending=False)
print(factor_df.head(5).to_string(index=False))

High-risk employee (test index 92):
  Predicted attrition probability: 100.0%
  Actual attrition: Yes

Top contributing factors:
             Feature  SHAP_value  Feature_value
            OverTime    1.858410            1.0
       MonthlyIncome    1.702828         1118.0
      JobInvolvement    1.270825            1.0
YearsWithCurrManager    0.896429            0.0
                 Age    0.808911           25.0


---
## 5. Low-Risk Employee Comparison
---

In [6]:
# Pick an employee with low predicted risk
low_risk_idx = np.argmin(probs)
low_risk_prob = probs[low_risk_idx]
print(f"Low-risk employee (test index {low_risk_idx}):")
print(f"  Predicted attrition probability: {low_risk_prob:.1%}")
print(f"  Actual attrition: {'Yes' if y_test.iloc[low_risk_idx] == 1 else 'No'}")
print()
shap_vals_low = shap_vals[low_risk_idx]
factor_low = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_value": shap_vals_low,
    "Feature_value": X_test.iloc[low_risk_idx].values
}).sort_values("SHAP_value", ascending=False)
print("Top protective factors (keep employee):")
print(factor_low.head(5).to_string(index=False))

Low-risk employee (test index 260):
  Predicted attrition probability: 0.0%
  Actual attrition: No

Top protective factors (keep employee):
                       Feature  SHAP_value  Feature_value
                   Gender_Male    0.112280            1.0
    JobRole_Research Scientist    0.083640            0.0
  EducationField_Life Sciences    0.041840            0.0
JobRole_Manufacturing Director    0.007562            1.0
          EducationField_Other    0.006375            0.0


---
## Explainability Summary

- **Global drivers of attrition:** OverTime, MonthlyIncome, JobSatisfaction, Age, YearsAtCompany
- **SHAP summary plot** saved as `models/shap_summary.png`
- **Feature importance bar** saved as `models/shap_importance_bar.png`
- **Local explanations** show per-employee contributing factors

HR can now see not just WHO is at risk, but WHY.

**Next step:** Model Versioning (09_model_versioning.ipynb)
